In [10]:
import moderngl
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from ipycanvas import Canvas, hold_canvas
from pyglm import glm
from plyfile import PlyData
from typing import Any, cast
import torch

In [ ]:
def splats_from_ply(path, device="cpu", requires_grad=True):
    vertex = PlyData.read(path)["vertex"]
    nsplats = len(vertex["x"])

    # 3 position + 1 opacity -> GLSL vec4 posopa
    mean3d = np.stack([vertex["x"], vertex["y"], vertex["z"]], axis=1).astype(np.float32)
    opacity = np.asarray(vertex["opacity"], dtype=np.float32).reshape(nsplats, 1)

    # vec4 scale；w 只是用于满足 std430 对齐
    scale = np.stack(
        [vertex["scale_0"], vertex["scale_1"], vertex["scale_2"]],
        axis=1,
    ).astype(np.float32)

    # PLY 中 rotation 为 (w, x, y, z)，与 shader 当前读取方式一致
    rot = np.stack([vertex[f"rot_{i}"] for i in range(4)], axis=1).astype(np.float32)

    # DC 是第 0 个 SH 系数，形状为 (N, 1, RGB)
    sh_dc = np.stack([vertex[f"f_dc_{i}"] for i in range(3)], axis=1).astype(np.float32)

    # f_rest 按 RGB 通道分别保存：先恢复 (N, 3, 15)，再转为 (N, 15, 3)
    sh_rest = np.stack([vertex[f"f_rest_{i}"] for i in range(45)], axis=1).astype(np.float32)

    # GLSL 是 vec4 shcolor[16]：RGB 后补一个 float，保证每个系数占 16 bytes
    sh = np.concatenate([sh_dc, sh_rest], axis=1)

    return (
        torch.tensor(mean3d, device=device, requires_grad=requires_grad),
        torch.tensor(opacity, device=device, requires_grad=requires_grad),
        torch.tensor(scale, device=device, requires_grad=requires_grad),
        torch.tensor(rot, device=device, requires_grad=requires_grad),
        torch.tensor(sh, device=device, requires_grad=requires_grad),
    )


class SplatData:
    device = "cpu"
    requires_grad = True

    def __init__(self, device="cpu", requires_grad=True) -> None:
        self.device = device
        self.requires_grad = requires_grad

    def from_ply(self, path):
        self.mean3d, self.opacity, self.scale, self.rot, self.sh = splats_from_ply(
            path, self.device, self.requires_grad
        )


class Scene:
    def __init__(
        self,
        _width,
        _height,
        _eye,
        _center,
        _up,
        _near,
        _far,
        _fovy,
        device="cpu",
        requires_grad=False,
        dtype=torch.float32,
    ) -> None:
        self._width = _width
        self._height = _height
        self._eye = _eye
        self._center = _center
        self._up = _up
        self._near = _near
        self._far = _far
        self._fovy = _fovy

        self.width = torch.tensor(_width, device=device, requires_grad=requires_grad, dtype=dtype)
        self.height = torch.tensor(_height, device=device, requires_grad=requires_grad, dtype=dtype)
        self.eye = torch.tensor(_eye, device=device, requires_grad=requires_grad, dtype=dtype)
        self.center = torch.tensor(_center, device=device, requires_grad=requires_grad, dtype=dtype)
        self.up = torch.tensor(_up, device=device, requires_grad=requires_grad, dtype=dtype)
        self.near = torch.tensor(_near, device=device, requires_grad=requires_grad, dtype=dtype)
        self.far = torch.tensor(_far, device=device, requires_grad=requires_grad, dtype=dtype)
        self.fovy = torch.tensor(_fovy, device=device, requires_grad=requires_grad, dtype=dtype)

        self.viewmtx = self._compute_viewmtx(
            device=device, requires_grad=requires_grad, dtype=dtype
        )
        self.projmtx = self._compute_projmtx(
            device=device, requires_grad=requires_grad, dtype=dtype
        )

    def _compute_viewmtx(self, device="cpu", requires_grad=False, dtype=torch.float32):
        m = glm.lookAt(self._eye, self._center, self._up)
        m = glm.transpose(m)
        return torch.tensor(m.to_list(), dtype=dtype, device=device, requires_grad=requires_grad)

    def _compute_projmtx(self, device="cpu", requires_grad=False, dtype=torch.float32):
        m = glm.perspective(self._fovy, self._width * 1.0 / self._height, self._near, self._far)
        m = glm.transpose(m)
        return torch.tensor(m.to_list(), dtype=dtype, device=device, requires_grad=requires_grad)


class GaussianSplat:
    def __init__(self, device="cpu") -> None:
        self.device = device
        self.SH_C0 = 0.28209479177387814
        self.SH_C1 = 0.4886025119029199
        self.SH_C2 = [
            1.0925484305920792,
            -1.0925484305920792,
            0.31539156525252005,
            -1.0925484305920792,
            0.5462742152960396,
        ]
        self.SH_C3 = [
            -0.5900435899266435,
            2.890611442640554,
            -0.4570457994644658,
            0.3731763325901154,
            -0.4570457994644658,
            1.445305721320277,
            -0.5900435899266435,
        ]

    def set_data(self, data: SplatData):
        self.data = data

    def quat2mtx(self, quat: torch.Tensor) -> torch.Tensor:
        q = quat / quat.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        w, x, y, z = q.unbind(dim=-1)
        # fmt: off
        R = torch.stack([
            1 - 2 * (y*y + z*z),     2 * (x*y - w*z),         2 * (x*z + w*y),
            2 * (x*y + w*z),         1 - 2 * (x*x + z*z),     2 * (y*z - w*x),
            2 * (x*z - w*y),         2 * (y*z + w*x),         1 - 2 * (x*x + y*y),
        ], dim=-1).reshape(-1,3,3)
        # fmt: on
        return R

    def compute_jacobian(self, scene: Scene, viewpos: torch.Tensor) -> torch.Tensor:
        x, y, z, _ = viewpos.unbind(dim=-2)
        z = z.clamp_max(-1e-8)  # TODO 验证 z 是小于0的
        z2 = z * z
        h = scene.height
        fovy = scene.fovy
        f = h / (2.0 * torch.tan(fovy * 0.5))
        fx = fy = f
        o = torch.zeros_like(x)
        # fmt: off
        J = torch.stack([
            fx / z, o,  -fx * x / z2,
            o,  fy / z, -fy * y / z2,
        ],dim=-1).reshape(-1, 2, 3)
        # fmt: on
        return J

    def compute_rect(self, pixelpos: torch.Tensor, Cov2d: torch.Tensor) -> torch.Tensor:
        u, v = pixelpos.unbind(dim=1)
        sxx = Cov2d[..., 0, 0]
        syy = Cov2d[..., 1, 1]
        sxy = Cov2d[..., 0, 1]
        trace = sxx + syy
        det = torch.clamp_min(sxx * syy - sxy * sxy, 1e-8)
        lambda_max = 0.5 * (trace + torch.sqrt(torch.clamp_min(trace * trace - 4.0 * det, 0.0)))
        r = torch.clamp(3.0 * torch.sqrt(lambda_max), 1.0, 1024.0)
        xmin = u - r
        xmax = u + r
        ymin = v - r
        ymax = v + r
        rect = torch.stack([xmin, ymin, xmax, ymax], dim=-1).squeeze(1)
        return rect

    def sh2rgb(self, sh: torch.Tensor, direct: torch.Tensor) -> torch.Tensor:
        """
        sh: (nspalts, 48)
        direct: (nspalts, 3)
        """
        x, y, z = direct.split(1, dim=-1)
        x2 = x * x
        y2 = y * y
        z2 = z * z
        xy = x * y
        yz = y * z
        xz = x * z

        nsplats = sh.size(0)
        sh = sh.view(nsplats, -1, 3)

        result = self.SH_C0 * sh[:, 0, :]

        result += self.SH_C1 * (-y * sh[:, 1, :] + z * sh[:, 2, :] - x * sh[:, 3, :])

        result += self.SH_C2[0] * sh[:, 4, :] * xy
        result += self.SH_C2[1] * sh[:, 5, :] * yz
        result += self.SH_C2[2] * sh[:, 6, :] * (2.0 * z2 - x2 - y2)
        result += self.SH_C2[3] * sh[:, 7, :] * xz
        result += self.SH_C2[4] * sh[:, 8, :] * (x2 - y2)

        result += self.SH_C3[0] * sh[:, 9, :] * y * (3.0 * x2 - y2)
        result += self.SH_C3[1] * sh[:, 10, :] * xy * z

        result += self.SH_C3[2] * sh[:, 11, :] * y * (4.0 * z2 - x2 - y2)
        result += self.SH_C3[3] * sh[:, 12, :] * z * (2.0 * z2 - 3.0 * x2 - 3.0 * y2)
        result += self.SH_C3[4] * sh[:, 13, :] * x * (4.0 * z2 - x2 - y2)
        result += self.SH_C3[5] * sh[:, 14, :] * z * (x2 - y2)
        result += self.SH_C3[6] * sh[:, 15, :] * x * (x2 - 3.0 * y2)

        result += 0.5
        result = result.clamp_min(0.0)
        return result

    def forward(self, scene: Scene):

        worldpos3 = self.data.mean3d
        worldpos4 = torch.cat([worldpos3, torch.ones_like(worldpos3[..., :1])], dim=-1)[..., None]

        W = scene.viewmtx
        P = scene.projmtx

        viewpos4 = W @ worldpos4
        ndcpos4 = P @ viewpos4
        ndcpos4.div_(ndcpos4[:, -1:, :])
        # TODO 删掉不在裁剪空间的splat
        pixpos = torch.stack(
            [
                (ndcpos4[:, 0, :] * 0.5 + 0.5) * scene.width,
                (0.5 - ndcpos4[:, 1, :] * 0.5) * scene.height,  # flip Y
            ],
            dim=-1,
        ).reshape(-1, 2)
        J = self.compute_jacobian(scene, viewpos4)
        S = torch.diag_embed(torch.exp(self.data.scale))
        R = self.quat2mtx(self.data.rot)
        Cov3d = R @ S @ S.mT @ R.mT
        W3 = W[0:3, 0:3]
        Cov3d = W3 @ Cov3d @ W3.mT
        Cov2d = J @ Cov3d @ J.mT
        rect = self.compute_rect(pixpos, Cov2d)

        # COLOR
        direct = worldpos4[:, 0:3, :].view(-1, 3) - scene.eye
        direct = direct / direct.norm(dim=-1, keepdim=True)
        rgb = self.sh2rgb(self.data.sh, direct)
        opacity = self.data.opacity
        color = torch.concat([rgb, opacity], dim=-1)

        # IMAGE
        image = torch.zeros(
            size=(scene._height, scene._width, 4), device=self.device, dtype=torch.float32
        )
        image[..., -1] = 1.0

        # pixposInt = pixpos.detach().floor().to(dtype=torch.long)
        # print(pixelpos2)
        # x = pixposInt[:, 0].squeeze(-1)
        # y = pixposInt[:, 1].squeeze(-1)

        # MASK
        # mask = (x >= 0) & (x < scene._width) & (y >= 0) & (y < scene._height)
        # print("mask.shape: ", mask.shape)
        # print("rectXmin.shape: ", rectXmin.shape)
        # rectXmin = rectXmin[mask]
        # rectYmin = rectYmin[mask]
        # rectXmax = rectXmax[mask]
        # rectYmax = rectYmax[mask]
        # color = color[mask]

        # image[y[mask], x[mask], :] = torch.tensor(
        #     [1, 1, 1, 1], device=self.device, dtype=torch.float32
        # )
        # image[y[mask], x[mask], :] = color

        # nspalts = rectXmin.size(0)
        # print("nspalts: ", nspalts)
        # for i in range(nspalts):
        #     if i % 1000 == 0:
        #         print(f"{i}/{nspalts}")
        #     if i > 11000:
        #         break
        #     image[rectYmin[i] : rectYmax[i], rectXmin[i] : rectXmax[i], :] = color[i]

        # image[rectYmin:rectYmax, rectXmin:rectXmax, :] = color

        self.fillRectByTile(image, pixpos, rect, Cov2d, color)

        return image.detach().cpu().numpy()

    def fillRectByTile(
        self,
        image: torch.Tensor,
        center: torch.Tensor,
        rect: torch.Tensor,
        cov2d: torch.Tensor,
        color: torch.Tensor,
    ):
        rectInt = rect.detach().floor().to(dtype=torch.long)

        # tile
        width = image.size(1)
        height = image.size(0)

        TILE_SIZE = 128
        CHUNK_SIZE = 128
        for txmin in range(0, width, TILE_SIZE):
            print(f"txmin: {txmin}, width: {width}")
            for tymin in range(0, height, TILE_SIZE):
                txmax = min(txmin + TILE_SIZE, width)
                tymax = min(tymin + TILE_SIZE, height)
                mask = self.maskByTile([txmin, tymin, txmax - 1, tymax - 1], rectInt)
                xsize = txmax - txmin
                ysize = tymax - tymin

                mrect = rectInt[mask]
                nsplats = rect.size(0)
                if nsplats == 0:
                    continue
                mcenter = center[mask]
                mcov2d = cov2d[mask].reshape(-1, 4)
                mcolor = color[mask]
                nsplats = mrect.size(0)
                for cmin in range(0, nsplats, CHUNK_SIZE):
                    cmax = min(cmin + CHUNK_SIZE, nsplats)
                    chunk = torch.concat(
                        [
                            mcenter[cmin:cmax, ...],
                            mrect[cmin:cmax, ...],
                            mcov2d[cmin:cmax, ...],
                            mcolor[cmin:cmax, ...],
                        ],
                        dim=-1,
                    )

                    print("chunk.shape: ", chunk.shape)
                    dx, dy = torch.meshgrid(
                        torch.arange(xsize, device=self.device),
                        torch.arange(ysize, device=self.device),
                        indexing="xy",
                    )
                    coords = torch.stack([txmin + dx, tymin + dy], dim=2).reshape(-1, 2)
                    pxchunk = torch.cat(
                        [
                            coords.to(dtype=torch.float32)[:, None, :].expand(
                                coords.size(0), chunk.size(0), coords.size(1)
                            ),
                            chunk[None, :, :].expand(coords.size(0), chunk.size(0), chunk.size(1)),
                        ],
                        dim=-1,
                    ).reshape(
                        -1, coords.size(1) + chunk.size(1)
                    )  # [pixel(2),center(2),rect(4),cov2d(4),color(4)]
                    weight = self.computeWeight(pxchunk[:, 0:2], pxchunk[:, 2:4], pxchunk[:, 4:8])
                    pxchunk = torch.cat([pxchunk, weight], dim=-1)
                    pxchunk = pxchunk[(weight < 0.001).squeeze(1)]
                    pxchunk[:, -2] = pxchunk[:, -2] * pxchunk[:, -1]  # opacity
                    pxchunk[:, -5:-2] = pxchunk[:, -5:-2] * pxchunk[:, -2:-1]  # rgb
                    coords = pxchunk[:, 0:2].to(dtype=torch.long)

                    for x in range(txmin, txmax):
                        print(f"x: {x}, txmax: {txmax}")
                        for y in range(tymin, tymax):
                            mask = (pxchunk[:, 0] == x) & (pxchunk[:, 1] == y)
                            ck = pxchunk[mask, :]
                            # print("ck.shape: ", ck.shape)

    def computeWeight(self, pixel: torch.Tensor, center: torch.Tensor, cov2d: torch.Tensor):
        d = pixel + 0.5 - center
        a = cov2d[:, 0]
        b = cov2d[:, 2]
        c = cov2d[:, 3]
        s = torch.clamp_min(a * c - b * b, 1e-6)
        x = d[:, 0]
        y = d[:, 1]
        r2 = (c * x * x - 2.0 * b * x * y + a * y * y) / s
        w = (r2 * -0.5).exp().unsqueeze(-1)
        return w

    def maskByTile(self, tileRect: list[int], splatRect: torch.Tensor):
        txmin, tymin, txmax, tymax = tileRect
        sxmin, symin, sxmax, symax = torch.unbind(splatRect, dim=1)
        xout = (sxmax < txmin) | (sxmin > txmax)
        yout = (symax < tymin) | (symin > tymax)
        return ~(xout | yout)

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"
splats = SplatData(device=device, requires_grad=True)
splats.from_ply("cactus_splat3.ply")
width = 512
height = 512
scene = Scene(
    _width=width,
    _height=height,
    _eye=[1, 1, 1],
    _center=[0, 0, 0],
    _up=[0, 1, 0],
    _near=0.01,
    _far=100,
    _fovy=glm.pi() / 2.0,
    device=device,
    requires_grad=False,
)
model = GaussianSplat(device=device)
model.set_data(splats)
image = model.forward(scene)
image = image * 255.0

print(image.shape)
canvas = Canvas(width=width, height=height)
with hold_canvas(canvas):
    canvas.clear()
    canvas.put_image_data(np.flipud(image))
canvas.put_image_data(image)
canvas

txmin: 0, width: 512
chunk.shape:  torch.Size([14, 14])
x: 0, txmax: 128
x: 1, txmax: 128
x: 2, txmax: 128
x: 3, txmax: 128
x: 4, txmax: 128
x: 5, txmax: 128
x: 6, txmax: 128
x: 7, txmax: 128
x: 8, txmax: 128
x: 9, txmax: 128
x: 10, txmax: 128
x: 11, txmax: 128
x: 12, txmax: 128
x: 13, txmax: 128
x: 14, txmax: 128
x: 15, txmax: 128
x: 16, txmax: 128
x: 17, txmax: 128
x: 18, txmax: 128
x: 19, txmax: 128
x: 20, txmax: 128
x: 21, txmax: 128
x: 22, txmax: 128
x: 23, txmax: 128
x: 24, txmax: 128
x: 25, txmax: 128
x: 26, txmax: 128
x: 27, txmax: 128
x: 28, txmax: 128
x: 29, txmax: 128
x: 30, txmax: 128
x: 31, txmax: 128
x: 32, txmax: 128
x: 33, txmax: 128
x: 34, txmax: 128
x: 35, txmax: 128
x: 36, txmax: 128
x: 37, txmax: 128
x: 38, txmax: 128
x: 39, txmax: 128
x: 40, txmax: 128
x: 41, txmax: 128
x: 42, txmax: 128
x: 43, txmax: 128
x: 44, txmax: 128
x: 45, txmax: 128
x: 46, txmax: 128
x: 47, txmax: 128
x: 48, txmax: 128
x: 49, txmax: 128
x: 50, txmax: 128
x: 51, txmax: 128
x: 52, txmax: 128


KeyboardInterrupt: 